# Mini-Challenge 3 — Refactor: from spaghetti to functions
### Saint Mary's · Wednesday lunch break

> *"Could you clean up that script? I want to reuse the high-risk filter for a different cohort."*

**Time:** 15 minutes · **Mode:** solo · **Goal:** see the difference between procedural code and modular code.

---

Below is a working but ugly script. **Your job:** refactor it into **three named functions**, then run them on the same data and verify you get the same output.

Pay attention to:
- Each function does **one** thing.
- Each function takes input as parameters, returns its result (no global state).
- Function names read like sentences: `parse_conditions(...)`, `filter_high_risk(...)`, `summarise(...)`.


# Solution — Mini-Challenge 3

In [ ]:
# Spaghetti script — works, but no one wants to maintain it.
# Run it once to confirm it works. Then refactor.

patients_raw = [
    {"id": "P201", "age": 72, "chronic": "diabetes_t2;hypertension"},
    {"id": "P202", "age": 45, "chronic": ""},
    {"id": "P203", "age": 81, "chronic": "copd;heart_failure;ckd"},
    {"id": "P204", "age": 33, "chronic": "asthma"},
    {"id": "P205", "age": 67, "chronic": "diabetes_t2"},
    {"id": "P206", "age": 90, "chronic": "heart_failure;ckd;dementia"},
    {"id": "P207", "age": 58, "chronic": "obesity;hypertension"},
]

# 1. parse chronic conditions into a list per patient
for p in patients_raw:
    if p["chronic"] == "":
        p["chronic_list"] = []
    else:
        p["chronic_list"] = p["chronic"].split(";")

# 2. flag high-risk: age >= 65 AND >= 2 chronic conditions
high_risk = []
for p in patients_raw:
    if p["age"] >= 65 and len(p["chronic_list"]) >= 2:
        high_risk.append(p)

# 3. summarise
print(f"Total patients: {len(patients_raw)}")
print(f"High-risk patients: {len(high_risk)}")
ages = [p["age"] for p in high_risk]
if ages:
    print(f"Mean age (high-risk): {sum(ages)/len(ages):.1f}")
else:
    print("No high-risk patients found.")

## Refactored version

In [ ]:
def parse_conditions(patient):
    """Add a 'chronic_list' field by splitting the 'chronic' string on ';'."""
    raw = patient.get("chronic", "")
    patient["chronic_list"] = raw.split(";") if raw else []
    return patient


def filter_high_risk(patients, min_age=65, min_chronic=2):
    """Return patients aged >= min_age with >= min_chronic conditions."""
    return [p for p in patients
            if p["age"] >= min_age and len(p["chronic_list"]) >= min_chronic]


def summarise(all_patients, high_risk):
    """Print the three KPIs."""
    print(f"Total patients: {len(all_patients)}")
    print(f"High-risk patients: {len(high_risk)}")
    ages = [p["age"] for p in high_risk]
    if ages:
        print(f"Mean age (high-risk): {sum(ages)/len(ages):.1f}")
    else:
        print("No high-risk patients found.")


patients_raw = [
    {"id": "P201", "age": 72, "chronic": "diabetes_t2;hypertension"},
    {"id": "P202", "age": 45, "chronic": ""},
    {"id": "P203", "age": 81, "chronic": "copd;heart_failure;ckd"},
    {"id": "P204", "age": 33, "chronic": "asthma"},
    {"id": "P205", "age": 67, "chronic": "diabetes_t2"},
    {"id": "P206", "age": 90, "chronic": "heart_failure;ckd;dementia"},
    {"id": "P207", "age": 58, "chronic": "obesity;hypertension"},
    {"id": "P208", "age": 75},   # missing 'chronic' key — handled by .get()
]

patients = [parse_conditions(p) for p in patients_raw]
high = filter_high_risk(patients)
summarise(patients, high)

## Why the refactor matters

- **`parse_conditions`** can now be tested in isolation, on one patient.
- **`filter_high_risk`** has parameters — we can reuse it with `min_age=80` for a different study.
- **`summarise`** doesn't know how patients were filtered. That's a feature.